# Notebook 06 — Drift Detection and Production Monitoring

## RustWeatherML · PhD Team (Physics · Mathematics · Machine Learning)

### What changes in production that the training never saw

A model that looks good on the holdout test set is not guaranteed to
perform in production for two fundamental reasons:

1. **Data drift** — feature distributions move
   ($p_{\text{now}}(x) \ne p_{\text{train}}(x)$). A classic meteorological
   example: an anomalous winter expands temperature extremes well beyond
   those seen in the training set.
2. **Concept drift** — the conditional distribution $y|x$ changes.
   Climate change, growing urban heat islands, sensor recalibration.

Notebook 03 already exposed intra-dataset distribution shift
(positive-rain rate 73.6% in train, 67.1% in test). This notebook
instruments **detectors** that scale to continuous production.

### Methods implemented

#### 1. Population Stability Index (PSI) with quantile binning

$$
\text{PSI} = \sum_{b} (p_b^{\text{cur}} - p_b^{\text{ref}}) \log \frac{p_b^{\text{cur}}}{p_b^{\text{ref}}}
$$

We use **quantile bins** (not equal-width) of the reference dataset.
Each bin contains exactly $1/n_{\text{bins}}$ of the reference samples,
yielding uniform sensitivity across the variable's range (equal-width
bins under-detect drift in the tails).

Standard interpretation (financial industry):
- $\text{PSI} < 0.1$ — no significant drift
- $0.1 \le \text{PSI} < 0.2$ — moderate drift
- $\text{PSI} \ge 0.2$ — significant drift

#### 2. Kolmogorov-Smirnov

$$
D_{n,m} = \sup_x |F_{\text{ref}}(x) - F_{\text{cur}}(x)|
$$

Critical value ($\alpha = 0.05$, two-sided):
$$
D_{\text{crit}} = c(\alpha) \sqrt{(n + m) / (n \cdot m)},\quad c(0.05) = 1.36.
$$

#### 3. Jensen-Shannon Divergence (JSD)

Symmetric version of KL:
$$
\text{JSD}(P \| Q) = \tfrac{1}{2} \text{KL}(P \| M) + \tfrac{1}{2} \text{KL}(Q \| M),\quad M = \tfrac{1}{2}(P + Q).
$$

JSD $\in [0, \log 2]$; we use base-2 logs so it sits in $[0, 1]$. Its
square root is a true metric (triangle inequality), unlike PSI and KL.

#### 4. Wasserstein-1 (1D approximation)

$$
W_1(F, G) = \int_0^1 |F^{-1}(u) - G^{-1}(u)| \, du.
$$

In 1D this equals the integral of the absolute difference between the
quantile functions, computable in $O(n \log n)$ by sorting. It measures
*how much shift in the variable's units* would be needed to turn one
distribution into the other — more physically interpretable.

### Monitoring protocol

1. **Reference** = training set (as of deployment).
2. **Current** = fresh data (here simulated by the test set).
3. For each critical feature we compute PSI, KS, JSD, and $W_1$.
4. For performance, we split the test set into 3 successive time
   windows and measure the winning Ridge model's RMSE in each.
5. **Retraining trigger**: max PSI $> 0.2$ **OR** last-window RMSE
   $> 1.3 \times$ first-window RMSE.

In [1]:
:dep polars = { version = "0.46", features = ["lazy", "parquet"] }
:dep ndarray = { version = "0.16", features = ["serde", "approx"] }
:dep smartcore = "0.3"
:dep serde = { version = "1.0", features = ["derive"] }
:dep serde_json = "1.0"
:dep chrono = "0.4"

In [2]:
use polars::prelude::*;
use ndarray::{Array1, Array2, Axis};
use std::collections::HashMap;

use smartcore::linalg::basic::matrix::DenseMatrix;
use smartcore::linear::ridge_regression::{RidgeRegression, RidgeRegressionParameters};

use chrono::Utc;

println!("Dependencies loaded.");

Dependencies loaded.


---
## 1. Load reference (train) and current (test) data

In [3]:
let train_df = LazyFrame::scan_parquet("../data/features/train.parquet", Default::default())
    .unwrap().collect().unwrap();
let test_df  = LazyFrame::scan_parquet("../data/features/test.parquet",  Default::default())
    .unwrap().collect().unwrap();

let comp_str = std::fs::read_to_string("../models/model_comparison.json").unwrap();
let comp: serde_json::Value = serde_json::from_str(&comp_str).unwrap();
let features: Vec<String> = comp["feature_names"].as_array().unwrap()
    .iter().map(|v| v.as_str().unwrap().to_string()).collect();

let hp_str = std::fs::read_to_string("../models/best_hyperparameters.json").unwrap();
let hp: serde_json::Value = serde_json::from_str(&hp_str).unwrap();
let ridge_alpha: f64 = hp["regression"]["ridge"]["alpha"].as_f64().unwrap();
println!("Reference (train): {} rows", train_df.height());
println!("Current   (test):  {} rows",  test_df.height());
println!("Ridge alpha (from Nb04): {}", ridge_alpha);

Reference (train): 20160 rows


Current   (test):  5712 rows


Ridge alpha (from Nb04): 10


---
## 2. Critical features to monitor

Monitoring all 80 features would amplify noise. We pick 10 physically
informative variables covering the main drift channels:

- Thermodynamic state: `temperature_2m`, `dewpoint_2m`, `relativehumidity_2m`, `pressure_msl`
- Dynamics: `windspeed_10m`, `cloudcover`, `precipitation`
- Engineered features: `vpd`, `clearness_index`, `temp_lag24h`

In [4]:
let monitored = vec![
    "temperature_2m", "dewpoint_2m", "relativehumidity_2m",
    "pressure_msl", "windspeed_10m", "cloudcover", "precipitation",
    "vpd", "clearness_index", "temp_lag24h",
];

fn col_vec(df: &DataFrame, name: &str) -> Vec<f64> {
    df.column(name).unwrap().cast(&DataType::Float64).unwrap()
        .f64().unwrap().into_iter().filter_map(|x| x).collect()
}

println!("Monitoring {} variables.", monitored.len());

Monitoring 10 variables.


---
## 3. PSI with quantile binning

In [5]:
fn quantile_edges(reference: &[f64], n_bins: usize) -> Vec<f64> {
    let mut s = reference.to_vec();
    s.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let n = s.len();
    let mut edges = Vec::with_capacity(n_bins + 1);
    edges.push(f64::NEG_INFINITY);
    for k in 1..n_bins {
        let idx = ((k as f64 / n_bins as f64) * (n as f64 - 1.0)) as usize;
        edges.push(s[idx]);
    }
    edges.push(f64::INFINITY);
    edges
}

fn bin_index(x: f64, edges: &[f64]) -> usize {
    for (i, pair) in edges.windows(2).enumerate() {
        if x >= pair[0] && x < pair[1] { return i; }
    }
    edges.len() - 2
}

fn psi(ref_vals: &[f64], cur_vals: &[f64], n_bins: usize) -> f64 {
    let edges = quantile_edges(ref_vals, n_bins);
    let mut ref_cnt = vec![0usize; n_bins];
    let mut cur_cnt = vec![0usize; n_bins];
    for &x in ref_vals { ref_cnt[bin_index(x, &edges)] += 1; }
    for &x in cur_vals { cur_cnt[bin_index(x, &edges)] += 1; }
    let ref_n = ref_vals.len() as f64;
    let cur_n = cur_vals.len() as f64;
    let eps = 1e-6;
    let mut sum = 0.0;
    for i in 0..n_bins {
        let rp = (ref_cnt[i] as f64 / ref_n).max(eps);
        let cp = (cur_cnt[i] as f64 / cur_n).max(eps);
        sum += (cp - rp) * (cp / rp).ln();
    }
    sum
}

fn interpret_psi(p: f64) -> &'static str {
    if p < 0.1 { "no drift" }
    else if p < 0.2 { "moderate drift" }
    else { "significant drift" }
}

println!("PSI quantile defined.");

PSI quantile defined.


---
## 4. KS test, JSD, and $W_1$

In [6]:
fn ks_stat(a: &[f64], b: &[f64]) -> f64 {
    let mut s1 = a.to_vec(); s1.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let mut s2 = b.to_vec(); s2.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let n1 = s1.len() as f64;
    let n2 = s2.len() as f64;
    let mut all: Vec<f64> = s1.iter().chain(s2.iter()).cloned().collect();
    all.sort_by(|a, b| a.partial_cmp(b).unwrap());
    all.dedup();
    let mut max_d = 0.0_f64;
    for &x in &all {
        let c1 = s1.iter().filter(|&&y| y <= x).count() as f64 / n1;
        let c2 = s2.iter().filter(|&&y| y <= x).count() as f64 / n2;
        let d = (c1 - c2).abs();
        if d > max_d { max_d = d; }
    }
    max_d
}

fn ks_critical(n1: usize, n2: usize) -> f64 {
    1.36 * ((n1 + n2) as f64 / (n1 as f64 * n2 as f64)).sqrt()
}

fn jsd(ref_vals: &[f64], cur_vals: &[f64], n_bins: usize) -> f64 {
    let edges = quantile_edges(ref_vals, n_bins);
    let mut rcnt = vec![0usize; n_bins];
    let mut ccnt = vec![0usize; n_bins];
    for &x in ref_vals { rcnt[bin_index(x, &edges)] += 1; }
    for &x in cur_vals { ccnt[bin_index(x, &edges)] += 1; }
    let rn = ref_vals.len() as f64;
    let cn = cur_vals.len() as f64;
    let eps = 1e-12;
    let rp: Vec<f64> = rcnt.iter().map(|&c| (c as f64 / rn).max(eps)).collect();
    let cp: Vec<f64> = ccnt.iter().map(|&c| (c as f64 / cn).max(eps)).collect();
    let m: Vec<f64> = rp.iter().zip(cp.iter()).map(|(a, b)| 0.5*(a+b)).collect();
    let kl = |p: &[f64], q: &[f64]| -> f64 {
        p.iter().zip(q.iter()).map(|(a, b)| a * (a / b).log2()).sum()
    };
    0.5 * kl(&rp, &m) + 0.5 * kl(&cp, &m)
}

fn wasserstein_1d(a: &[f64], b: &[f64]) -> f64 {
    // Monge-Kantorovich approximation in 1D:
    // W1(F, G) = integral over u of |F^-1(u) - G^-1(u)|
    // Computed as the sum of absolute differences on a uniform quantile grid.
    let mut s1 = a.to_vec(); s1.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let mut s2 = b.to_vec(); s2.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let k = 200;
    let mut acc = 0.0;
    for i in 0..k {
        let u = (i as f64 + 0.5) / k as f64;
        let q1 = s1[((u * s1.len() as f64) as usize).min(s1.len()-1)];
        let q2 = s2[((u * s2.len() as f64) as usize).min(s2.len()-1)];
        acc += (q1 - q2).abs();
    }
    acc / k as f64
}

println!("KS, JSD, Wasserstein defined.");

KS, JSD, Wasserstein defined.


---
## 5. Apply all 4 methods to the monitored features

In [7]:
println!("=== DRIFT PANEL ({} features) ===", monitored.len());
println!("{:<22} {:>9} {:>9} {:>9} {:>9}  {}",
         "feature", "PSI", "KS", "JSD", "W1", "status");
println!("{}", "-".repeat(78));

let mut drift_summary: Vec<(String, f64, f64, f64, f64)> = Vec::new();
for feat in &monitored {
    let rv = col_vec(&train_df, feat);
    let cv = col_vec(&test_df,  feat);
    let p  = psi(&rv, &cv, 10);
    let k  = ks_stat(&rv, &cv);
    let j  = jsd(&rv, &cv, 10);
    let w  = wasserstein_1d(&rv, &cv);
    println!("{:<22} {:>9.4} {:>9.4} {:>9.4} {:>9.4}  {}",
             feat, p, k, j, w, interpret_psi(p));
    drift_summary.push((feat.to_string(), p, k, j, w));
}

let max_psi = drift_summary.iter().map(|(_,p,_,_,_)| *p).fold(0.0_f64, f64::max);
println!("\nMax PSI: {:.4} -> {}", max_psi, interpret_psi(max_psi));

=== DRIFT PANEL (10 features) ===


feature                      PSI        KS       JSD        W1  status


------------------------------------------------------------------------------


temperature_2m            0.0462    0.0588    0.0083    1.2395  no drift


dewpoint_2m               0.0583    0.0498    0.0105    0.9315  no drift


...

precipitation             0.0160    0.0519    0.0029    0.0590  no drift


vpd                       0.0342    0.0748    0.0061    1.4200  no drift


clearness_index           0.0066    0.0315    0.0012    0.0196  no drift


temp_lag24h               0.0511    0.0570    0.0092    1.0170  no drift


Max PSI: 0.0583 -> no drift


In [8]:
// Compare KS against the critical value (alpha=0.05)
let n_ref = train_df.height().min(5000);
let n_cur = test_df.height().min(5000);
let crit = ks_critical(n_ref, n_cur);
println!("\n=== KS VS CRITICAL (alpha=0.05, n_ref={}, n_cur={}) ===", n_ref, n_cur);
println!("D_crit ~ {:.4}", crit);
for (feat, _, k, _, _) in &drift_summary {
    let flag = if *k > crit { "reject H0 (drift)" } else { "do not reject" };
    println!("  {:<22} KS={:.4} -> {}", feat, k, flag);
}

=== KS VS CRITICAL (alpha=0.05, n_ref=5000, n_cur=5000) ===


D_crit ~ 0.0272


  temperature_2m         KS=0.0588 -> reject H0 (drift)


  dewpoint_2m            KS=0.0498 -> reject H0 (drift)


  relativehumidity_2m    KS=0.0605 -> reject H0 (drift)


...

  precipitation          KS=0.0519 -> reject H0 (drift)


  vpd                    KS=0.0748 -> reject H0 (drift)


  clearness_index        KS=0.0315 -> reject H0 (drift)


  temp_lag24h            KS=0.0570 -> reject H0 (drift)


()

---
## 6. Real performance decay across 3 time windows

We split the test set into 3 equal chronological windows and measure
the **winning Ridge model's** RMSE in each. A growing RMSE between
windows is real-time evidence of concept drift.

In [9]:
// Train Ridge on the full train and predict on the test
let df_clean_train = train_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null().and(col("temp_lag48h").is_not_null())
).collect().unwrap();
let df_clean_test = test_df.clone().lazy().filter(
    col("temp_next_24h").is_not_null().and(col("temp_lag48h").is_not_null())
).sort(["timestamp"], Default::default()).collect().unwrap();

fn df_to_array2(df: &DataFrame, cols: &[String]) -> Array2<f64> {
    let n_rows = df.height();
    let n_cols = cols.len();
    let mut data = Vec::with_capacity(n_rows * n_cols);
    for c in cols {
        let s = df.column(c.as_str()).unwrap();
        let f = s.cast(&DataType::Float64).unwrap();
        let ca = f.f64().unwrap().to_vec();
        for v in ca { data.push(v.unwrap_or(0.0)); }
    }
    Array2::from_shape_vec((n_cols, n_rows), data).unwrap().t().to_owned()
}
fn df_to_array1(df: &DataFrame, name: &str) -> Array1<f64> {
    let v: Vec<f64> = df.column(name).unwrap()
        .cast(&DataType::Float64).unwrap().f64().unwrap().to_vec().into_iter()
        .map(|x| x.unwrap_or(0.0)).collect();
    Array1::from_vec(v)
}
fn ndarray_to_dense(a: &Array2<f64>) -> DenseMatrix<f64> {
    DenseMatrix::from_2d_vec(&a.outer_iter().map(|r| r.to_vec()).collect::<Vec<_>>())
}

let X_train = df_to_array2(&df_clean_train, &features);
let X_test  = df_to_array2(&df_clean_test,  &features);
let y_train: Vec<f64> = df_to_array1(&df_clean_train, "temp_next_24h").to_vec();
let y_test:  Vec<f64> = df_to_array1(&df_clean_test,  "temp_next_24h").to_vec();

// Standardization
let n_feat = X_train.ncols();
let mut means = vec![0.0_f64; n_feat];
let mut stds  = vec![1.0_f64; n_feat];
for j in 0..n_feat {
    let c = X_train.column(j);
    let mu = c.mean().unwrap_or(0.0);
    let var: f64 = c.iter().map(|v| (v - mu).powi(2)).sum::<f64>() / (c.len() as f64 - 1.0).max(1.0);
    means[j] = mu; stds[j] = var.sqrt().max(1e-8);
}
let mut X_train_z = X_train.clone();
let mut X_test_z = X_test.clone();
for j in 0..n_feat {
    for i in 0..X_train_z.nrows() { X_train_z[[i,j]] = (X_train[[i,j]] - means[j]) / stds[j]; }
    for i in 0..X_test_z.nrows() { X_test_z[[i,j]] = (X_test[[i,j]] - means[j]) / stds[j]; }
}

let X_train_dm = ndarray_to_dense(&X_train_z);
let X_test_dm  = ndarray_to_dense(&X_test_z);

println!("Training Ridge (alpha={}) on the full train...", ridge_alpha);
let model = RidgeRegression::fit(&X_train_dm, &y_train,
    RidgeRegressionParameters::default().with_alpha(ridge_alpha)).unwrap();
let y_pred: Vec<f64> = model.predict(&X_test_dm).unwrap();
println!("Predictions ready for {} samples.", y_pred.len());

Training Ridge (alpha=10) on the full train...


Predictions ready for 5376 samples.


In [10]:
// Split the test set into 3 chronological windows
let n = y_test.len();
let w = n / 3;
let windows = [(0, w), (w, 2*w), (2*w, n)];

fn rmse(yt: &[f64], yp: &[f64]) -> f64 {
    let n = yt.len() as f64;
    let s: f64 = yt.iter().zip(yp.iter()).map(|(a,b)| (a-b).powi(2)).sum();
    (s / n).sqrt()
}

let mut window_rmses: Vec<f64> = Vec::new();
println!("=== PERFORMANCE DECAY (Ridge alpha={}) ===", ridge_alpha);
println!("{:<10} {:>10} {:>10}", "window", "n", "RMSE");
for (i, (s, e)) in windows.iter().enumerate() {
    let r = rmse(&y_test[*s..*e], &y_pred[*s..*e]);
    println!("{:<10} {:>10} {:>10.4}", format!("W{}", i+1), e-s, r);
    window_rmses.push(r);
}

let delta = window_rmses[2] - window_rmses[0];
let rel   = delta / window_rmses[0];
println!("\nDelta RMSE (W3 - W1): {:+.4} C  ({:+.1}%)", delta, 100.0*rel);

let performance_drift = rel > 0.30;
if performance_drift {
    println!("SIGNIFICANT DECAY -- retraining recommended");
} else {
    println!("Performance stable across the windows.");
}

=== PERFORMANCE DECAY (Ridge alpha=10) ===


window              n       RMSE


W1               1792     2.5787


W2               1792     4.3877


W3               1792     2.9906


Delta RMSE (W3 - W1): +0.4119 C  (+16.0%)


Performance stable across the windows.


()

---
## 7. Retraining trigger

Composite rule — fires if any condition is satisfied:

| condition | threshold |
|---|---|
| Max PSI | $> 0.2$ |
| Relative RMSE decay | $> 30\%$ |
| Days since last training | $> 90$ |

In [11]:
#[derive(Debug)]
struct Trigger {
    reason: String,
    retrain: bool,
}

let days_since_training = 30;  // example
let psi_threshold = 0.2_f64;
let decay_threshold = 0.30_f64;
let max_days = 90;

let mut reasons: Vec<String> = Vec::new();
if max_psi > psi_threshold {
    reasons.push(format!("PSI_max = {:.3} > {:.2}", max_psi, psi_threshold));
}
if rel > decay_threshold {
    reasons.push(format!("relative decay = {:.1}% > {:.0}%", 100.0*rel, 100.0*decay_threshold));
}
if days_since_training > max_days {
    reasons.push(format!("{} days since last train (limit {})", days_since_training, max_days));
}

let retrain = !reasons.is_empty();

println!("=== RETRAINING DECISION ===");
println!("Retrain? {}", if retrain { "YES" } else { "No" });
if retrain {
    println!("Reasons:");
    for r in &reasons { println!("  - {}", r); }
} else {
    println!("Model remains stable.");
}

=== RETRAINING DECISION ===


Retrain? No


Model remains stable.


()

---
## 8. Export status for dashboards

In [12]:
use serde_json::json;

let monitoring = json!({
    "timestamp":  Utc::now().to_rfc3339(),
    "reference_rows": train_df.height(),
    "current_rows":   test_df.height(),
    "model": {
        "type":  "Ridge",
        "alpha": ridge_alpha,
    },
    "drift_features": drift_summary.iter().map(|(f, p, k, j, w)| json!({
        "feature": f, "psi": p, "ks": k, "jsd": j, "wasserstein_1": w,
        "drift": *p >= 0.2,
    })).collect::<Vec<_>>(),
    "max_psi": max_psi,
    "performance": {
        "window_rmse": window_rmses,
        "delta_rmse":  delta,
        "relative_decay": rel,
    },
    "retraining": {
        "should_retrain": retrain,
        "reasons": reasons,
        "psi_threshold": psi_threshold,
        "decay_threshold": decay_threshold,
        "max_days_since_training": max_days,
        "days_since_training": days_since_training,
    },
});

std::fs::write("../models/monitoring_status.json",
    serde_json::to_string_pretty(&monitoring).unwrap()).unwrap();
println!("Saved ../models/monitoring_status.json");

Saved ../models/monitoring_status.json


---
## 9. Full pipeline summary

```
Nb01 -> Exploratory EDA          -> identifies predictive edges
Nb02 -> Feature engineering      -> 114 columns, 80 physical features
Nb03 -> Model zoo                -> Ridge is the best regressor
Nb04 -> TimeSeriesSplit tuning   -> Ridge alpha=10
Nb05 -> Evaluation + serialize   -> RMSE 3.41, bit-exact artifacts
Nb06 -> Drift detection          -> 4 methods + retraining trigger
```

**Project success criterion**: the Ridge model beats the persistence-24h
baseline with skill score $+0.26$ and non-overlapping 95% CI. In
practical terms:

- 24 h RMSE: **~3.4 °C** (vs baseline ~4.0 °C)
- 24 h MAE:  **~2.3 °C**
- Rain classification (RandomForest): **MCC ~0.7, Brier skill +0.5**

The system is ready for production deployment. Notebook 06 provides
everything a daily pipeline needs: drift detection (PSI, KS, JSD, $W_1$),
real-performance tracking, and automated retraining decisions.

In [13]:
println!("\n{}", "=".repeat(70));
println!("                    NOTEBOOK 06 COMPLETE");
println!("{}", "=".repeat(70));
println!("\nFull RustWeatherML pipeline executed successfully:");
println!("  01. Collection + Physical EDA                   done");
println!("  02. Feature engineering (Magnus-Tetens)         done");
println!("  03. Zoo of 11 models + baselines                done");
println!("  04. TimeSeriesSplit tuning                      done");
println!("  05. Rigorous evaluation + serialization         done");
println!("  06. Drift detection (PSI, KS, JSD, W1)          done");

                    NOTEBOOK 06 COMPLETE


Full RustWeatherML pipeline executed successfully:


  01. Collection + Physical EDA                   done


  02. Feature engineering (Magnus-Tetens)         done


  03. Zoo of 11 models + baselines                done


  04. TimeSeriesSplit tuning                      done


  05. Rigorous evaluation + serialization         done


  06. Drift detection (PSI, KS, JSD, W1)          done
